In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/customer-churn-training.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Dataset shape: (12, 7)
Columns: ['customer_id', 'tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type', 'churned']


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


In [3]:
# 1. Check data types
print("DATA TYPES")
display(df.dtypes.to_frame("Data Type"))

# 2. Check missing values
print("\nMISSING VALUES")
display(df.isnull().sum().to_frame("Missing Count"))

# 3. Check duplicate rows
print("\nDUPLICATE ROWS")
print("Total duplicates:", df.duplicated().sum())

# 4. Check duplicate customer IDs
print("\nDUPLICATE CUSTOMER IDs")
print("Duplicate IDs:", df["customer_id"].duplicated().sum())

# 5. Check target class balance
print("\nCHURN CLASS DISTRIBUTION")
display(df["churned"].value_counts().to_frame("Count"))

print("\nCHURN CLASS PERCENTAGE")
display(
    (df["churned"].value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("Percentage")
)

# 6. Check numerical statistics
print("\nNUMERICAL SUMMARY")
display(df.describe().T)

# 7. Check categorical values
print("\nPLAN TYPE DISTRIBUTION")
display(df["plan_type"].value_counts(dropna=False))

DATA TYPES


,Data Type
customer_id,object
tenure_months,int64
support_tickets,int64
monthly_spend_inr,int64
last_login_days,int64
plan_type,object
churned,int64



MISSING VALUES


,Missing Count
customer_id,0
tenure_months,0
support_tickets,0
monthly_spend_inr,0
last_login_days,0
plan_type,0
churned,0



DUPLICATE ROWS
Total duplicates: 0

DUPLICATE CUSTOMER IDs
Duplicate IDs: 0

CHURN CLASS DISTRIBUTION


...

,Percentage
churned,
0,58.33
1,41.67



NUMERICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
tenure_months,12.0,12.166667,9.242524,1.0,5.50,9.5,18.50,30.0
support_tickets,12.0,2.250000,1.602555,0.0,1.00,2.0,3.25,5.0
monthly_spend_inr,12.0,915.666667,332.574895,499.0,724.00,899.0,1074.00,1499.0
last_login_days,12.0,10.000000,9.095453,1.0,2.75,7.0,14.75,30.0
churned,12.0,0.416667,0.514929,0.0,0.00,0.0,1.00,1.0



PLAN TYPE DISTRIBUTION


plan_type
Basic       5
Standard    4
Pro         3
Name: count, dtype: int64

In [4]:
# STEP 3: INVALID VALUES, OUTLIERS & LEAKAGE CHECKS

print("1. INVALID VALUE CHECKS")

print("Negative tenure:",
      (df["tenure_months"] < 0).sum())

print("Negative support tickets:",
      (df["support_tickets"] < 0).sum())

print("Non-positive monthly spend:",
      (df["monthly_spend_inr"] <= 0).sum())

print("Negative last login days:",
      (df["last_login_days"] < 0).sum())

print("Invalid churn labels:",
      (~df["churned"].isin([0, 1])).sum())

print("\n2. PLAN TYPE VALIDATION")

print("Unique plan types:", df["plan_type"].unique())

print("\n3. OUTLIER CHECK USING IQR")

numeric_columns = [
    "tenure_months",
    "support_tickets",
    "monthly_spend_inr",
    "last_login_days"
]

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}: {len(outliers)} potential outlier(s)")
    if not outliers.empty:
        display(outliers[["customer_id", col]])

print("\n4. IDENTIFIER CHECK")

print("Customer IDs are unique:",
      df["customer_id"].is_unique)

print("\n5. TARGET AND FEATURE SEPARATION")

TARGET = "churned"
ID_COLUMN = "customer_id"

X = df.drop(columns=[TARGET, ID_COLUMN])
y = df[TARGET]

print("Feature columns:", X.columns.tolist())
print("Target column:", TARGET)
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

1. INVALID VALUE CHECKS
Negative tenure: 0
Negative support tickets: 0
Non-positive monthly spend: 0
Negative last login days: 0
Invalid churn labels: 0

2. PLAN TYPE VALIDATION
Unique plan types: ['Basic' 'Pro' 'Standard']

3. OUTLIER CHECK USING IQR
tenure_months: 0 potential outlier(s)
support_tickets: 0 potential outlier(s)
monthly_spend_inr: 0 potential outlier(s)
last_login_days: 0 potential outlier(s)

4. IDENTIFIER CHECK
Customer IDs are unique: True

5. TARGET AND FEATURE SEPARATION
Feature columns: ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type']
Target column: churned
Feature shape: (12, 5)
Target shape: (12,)


In [5]:

# STEP 6: MAJORITY-CLASS BASELINE MODEL

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# 1. Define features and target
TARGET = "churned"
ID_COLUMN = "customer_id"

X = df.drop(columns=[TARGET, ID_COLUMN]).copy()
y = df[TARGET].copy()

# 2. Split data BEFORE fitting any learned preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

# 3. Create the majority-class baseline
baseline = DummyClassifier(strategy="most_frequent")

# DummyClassifier does not learn from feature values
baseline.fit(X_train, y_train)

# 4. Predict test labels
y_pred = baseline.predict(X_test)

# 5. Evaluate baseline
print("\nBASELINE EVALUATION")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(
    precision_score(y_test, y_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_test, y_pred, zero_division=0), 4
))
print("F1-score:", round(
    f1_score(y_test, y_pred, zero_division=0), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Non-churn (0)", "Churn (1)"],
    zero_division=0
))


Training records: 9
Testing records: 3

Training target distribution:
churned
0    5
1    4
Name: count, dtype: int64

Testing target distribution:
churned
0    2
1    1
Name: count, dtype: int64

BASELINE EVALUATION
Accuracy: 0.6667
Precision: 0.0
Recall: 0.0
F1-score: 0.0

Confusion Matrix:
[[2 0]
 [1 0]]

Classification Report:
               precision    recall  f1-score   support

Non-churn (0)       0.67      1.00      0.80         2
    Churn (1)       0.00      0.00      0.00         1

     accuracy                           0.67         3
    macro avg       0.33      0.50      0.40         3
 weighted avg       0.44      0.67      0.53         3



In [6]:

# STEP 9: FINAL PROBLEM FRAMING SUMMARY

problem_framing = {
    "business_problem": (
        "Understand customer churn patterns and investigate "
        "whether available customer attributes can help identify churn."
    ),
    "ml_problem": (
        "Binary classification with 'churned' as the target variable."
    ),
    "target_variable": "churned",
    "excluded_identifier": "customer_id",
    "dataset_records": int(len(df)),
    "input_features": int(df.drop(
        columns=["churned", "customer_id"]
    ).shape[1]),
    "baseline_model": "DummyClassifier(strategy='most_frequent')",
    "baseline_accuracy": 0.6667,
    "baseline_churn_precision": 0.0,
    "baseline_churn_recall": 0.0,
    "baseline_churn_f1": 0.0,
    "primary_evaluation_metric": "Churn-class recall, alongside precision and F1",
    "deployment_ready": False,
    "main_limitation": (
        "Only 12 records are available; the 3-record test set "
        "is too small for reliable performance conclusions."
    )
}

for key, value in problem_framing.items():
    print(f"{key}: {value}")


business_problem: Understand customer churn patterns and investigate whether available customer attributes can help identify churn.
ml_problem: Binary classification with 'churned' as the target variable.
target_variable: churned
excluded_identifier: customer_id
dataset_records: 12
input_features: 5
baseline_model: DummyClassifier(strategy='most_frequent')
baseline_accuracy: 0.6667
baseline_churn_precision: 0.0
baseline_churn_recall: 0.0
baseline_churn_f1: 0.0
primary_evaluation_metric: Churn-class recall, alongside precision and F1
deployment_ready: False
main_limitation: Only 12 records are available; the 3-record test set is too small for reliable performance conclusions.
